# Conditional Image Synthesis — Exploration Notebook

This notebook lets you:
- Load a trained model and generate images from segmentation maps
- Condition on text prompts via CLIP
- Interpolate between text prompts
- Visualize SPADE gate activations
- Compare text-conditioned vs unconditioned outputs


In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from torchvision.utils import make_grid
from PIL import Image

from models.generator    import UNetGenerator
from models.clip_encoder import CLIPTextEncoder, CLIPProjector, NullTextEmbedding
from data.datasets       import get_seg_transform, seg_to_onehot, denormalize_image, colorize_seg
from utils.training      import EMA, load_checkpoint

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load trained model

In [ ]:
CHECKPOINT    = '../outputs/cityscapes/ckpts/best.pt'
NUM_CLASSES   = 35
IMAGE_SIZE    = 256
BASE_CHANNELS = 64
CLIP_DIM      = 512
Z_DIM         = 256

generator = UNetGenerator(
    image_size=IMAGE_SIZE, in_channels=3, num_seg_classes=NUM_CLASSES,
    base_channels=BASE_CHANNELS, channel_mults=(1,2,4,8,8),
    clip_dim=CLIP_DIM, attn_levels=(2,3), z_dim=Z_DIM,
).to(device)

from models.discriminator import MultiScaleDiscriminator
discriminator = MultiScaleDiscriminator(image_channels=3, seg_channels=NUM_CLASSES).to(device)

ema = EMA(generator)
load_checkpoint(CHECKPOINT, generator, discriminator, ema=ema, device=device)
ema.apply_shadow(generator)
generator.eval()

clip_encoder = CLIPTextEncoder('ViT-B-32', device=device)
clip_proj    = CLIPProjector(clip_encoder.embed_dim, CLIP_DIM).to(device)
null_emb     = NullTextEmbedding(CLIP_DIM, clip_encoder.max_tokens).to(device)
print('Model loaded!')

## 2. Generate from a single segmentation map + text

In [ ]:
seg_path = '../data/cityscapes/gtFine/val/frankfurt/frankfurt_000000_000294_gtFine_labelIds.png'
seg_img  = Image.open(seg_path)
tfm = get_seg_transform(IMAGE_SIZE, augment=False)
seg = tfm(seg_img)
seg_onehot = seg_to_onehot(seg, NUM_CLASSES).unsqueeze(0).to(device)

texts = [
    None,
    'a sunny summer day in the city',
    'a rainy night with wet roads',
    'heavy snowfall on city streets',
    'golden sunset over the city',
]

outputs = [colorize_seg(seg_onehot.cpu(), NUM_CLASSES)[0]]   # Seg reference

torch.manual_seed(42)
z = torch.randn(1, Z_DIM, device=device)

with torch.no_grad():
    for text in texts:
        if text:
            token_emb, _ = clip_encoder.encode_text([text])
            token_emb = clip_proj(token_emb.to(device))
        else:
            token_emb, _ = null_emb(1, device)
            token_emb = clip_proj(token_emb)
        fake = generator(seg_onehot, token_emb, z)
        outputs.append(denormalize_image(fake.cpu())[0])

fig, axes = plt.subplots(1, len(outputs), figsize=(4 * len(outputs), 4))
titles = ['Seg Map'] + ['No text'] + [t[:20]+'...' if t and len(t)>20 else (t or '') for t in texts[1:]]
for ax, img, title in zip(axes, outputs, titles):
    ax.imshow(img.permute(1,2,0).clamp(0,1).numpy())
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Same seg map, different text conditioning', fontsize=12)
plt.tight_layout()
plt.savefig('text_conditioning.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Text embedding interpolation

In [ ]:
text_a = 'a bright sunny day'
text_b = 'a dark stormy night'
n_steps = 8

with torch.no_grad():
    tok_a, _ = clip_encoder.encode_text([text_a])
    tok_b, _ = clip_encoder.encode_text([text_b])
    tok_a = clip_proj(tok_a.to(device))
    tok_b = clip_proj(tok_b.to(device))

    torch.manual_seed(0)
    z = torch.randn(1, Z_DIM, device=device)
    interp_imgs = []
    for i in range(n_steps):
        alpha = i / (n_steps - 1)
        tok_interp = (1 - alpha) * tok_a + alpha * tok_b
        fake = generator(seg_onehot, tok_interp, z)
        interp_imgs.append(denormalize_image(fake.cpu())[0])

grid = make_grid(torch.stack(interp_imgs), nrow=n_steps)
plt.figure(figsize=(20, 3))
plt.imshow(grid.permute(1,2,0).clamp(0,1).numpy())
plt.title(f'Text interpolation: "{text_a}" → "{text_b}"', fontsize=12)
plt.axis('off')
plt.savefig('text_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Noise (style) variations — same seg + text, different z

In [ ]:
text = 'a rainy evening in the city'
n_var = 8

with torch.no_grad():
    token_emb, _ = clip_encoder.encode_text([text])
    token_emb = clip_proj(token_emb.to(device))

    variations = []
    for i in range(n_var):
        torch.manual_seed(i)
        z = torch.randn(1, Z_DIM, device=device)
        fake = generator(seg_onehot, token_emb, z)
        variations.append(denormalize_image(fake.cpu())[0])

grid = make_grid(torch.stack(variations), nrow=n_var)
plt.figure(figsize=(20, 3))
plt.imshow(grid.permute(1,2,0).clamp(0,1).numpy())
plt.title(f'Style variations for: "{text}"', fontsize=12)
plt.axis('off')
plt.savefig('style_variations.png', dpi=150, bbox_inches='tight')
plt.show()